# Trading TD v3 + Ensemble -- Google Colab (Cella unica, avvio singolo)
**H4+H1+M15 | Gate G1-G6 | Score confluenza 0-100 | 40 strumenti piano free**

### Istruzioni
1. API Key Twelvedata gia' salvata nei **Colab Secrets** come `TWELVEDATA_API_KEY` (fatto).
2. Clicca **una sola volta** il pulsante ▶ della cella di codice qui sotto (oppure Ctrl+Invio dentro la cella).
3. Alla prima esecuzione della sessione, Google chiedera' l'autorizzazione a montare il Drive: e' un prompt di sicurezza obbligatorio di Google, va confermato una volta. Dopo la conferma, tutto il resto procede da solo.
4. Al termine (~20-25 min con piano free), l'unico file di output **Ensemble_H4H1M15_<timestamp>.xlsx** viene salvato automaticamente in `Il tuo Drive/ENSEMBLE/`.

> Score confluenza: indicatore tecnico, NON probabilita' statistica validata.


In [ ]:
# -- Installazione dipendenze + imports
# +==============================================================+
# |  TRADING ANALYSIS TD v3 + ENSEMBLE -- Google Colab            |
# |  H4 + H1 + M15 ? Gate G1-G6 ? Score S1-S6 ? Ensemble        |
# |                                                              |
# |  ISTRUZIONI:                                                 |
# |  1. Apri in Google Colab (File -> Carica notebook -> .py)     |
# |  2. Inserisci la API Key in CONFIGURAZIONE (cella 2)         |
# |     oppure salvala in Colab Secrets come TWELVEDATA_API_KEY  |
# |  3. Esegui tutto (Runtime -> Esegui tutto)                    |
# |  4. I file Excel vengono scaricati automaticamente           |
# +==============================================================+

# --------------------------------------------------------------
# CELLA 1 -- Installazione dipendenze
# --------------------------------------------------------------
import subprocess, sys
pkgs = ["requests", "pandas", "openpyxl", "pytz", "numpy"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("OK Dipendenze installate")

# --------------------------------------------------------------
# CELLA 2 -- Importazioni e rilevamento ambiente
# --------------------------------------------------------------
import time, requests, re, glob, os, warnings
import pandas as pd

#==============================================================

# -- CONFIGURAZIONE -- API Key
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime, timezone
import pytz
warnings.filterwarnings("ignore")

# Rileva Google Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("OK Google Colab rilevato -- i file Excel verranno scaricati automaticamente")
except ImportError:
    IN_COLAB = False
    print("??  Ambiente locale -- i file verranno salvati nella cartella corrente")

WORK_DIR = "/content" if IN_COLAB else os.getcwd()

# --------------------------------------------------------------
# Mount Google Drive + cartella output ENSEMBLE (esecuzione automatica)
# --------------------------------------------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ENSEMBLE_DIR = "/content/drive/MyDrive/ENSEMBLE"
    os.makedirs(DRIVE_ENSEMBLE_DIR, exist_ok=True)
    print(f"OK Google Drive montato -- output ensemble in: {DRIVE_ENSEMBLE_DIR}")
else:
    DRIVE_ENSEMBLE_DIR = WORK_DIR
    print(f"?? Ambiente locale -- output ensemble in: {DRIVE_ENSEMBLE_DIR}")

# --------------------------------------------------------------
# CELLA 3 -- CONFIGURAZIONE (modifica qui)
# --------------------------------------------------------------

# API Key Twelvedata -- due opzioni:
# Opzione A (consigliata): salva in Colab Secrets come "TWELVEDATA_API_KEY"
# Opzione B: inserisci direttamente tra le virgolette
try:
    from google.colab import userdata
    API_KEY = userdata.get("TWELVEDATA_API_KEY")
    print(f"OK API Key caricata da Colab Secrets ({API_KEY[:4]}...{API_KEY[-4:]})")
except Exception:
    API_KEY = os.environ.get("TWELVEDATA_API_KEY", "c71a0ee8031340cfaff06641d39dcfef")   # <- Opzione B: inserisci qui

if API_KEY == "LA_TUA_API_KEY" or not API_KEY or not API_KEY.strip():
    raise ValueError(
        "(X) API Key non configurata.\n"
        "   Opzione A: aggiungi TWELVEDATA_API_KEY nei Colab Secrets (? icona a sinistra)\n"
        "   Opzione B: modifica la riga API_KEY = '...' in questa cella"
    )

BASE_URL      = "https://api.twelvedata.com/time_series"
ROME_TZ       = pytz.timezone("Europe/Rome")
PLATFORM      = "Twelvedata"
RR_MIN        = 2.5
REQUEST_DELAY = 9.0    # secondi tra call API (piano free: 8 req/min)
SCORE_MIN_ENS = 3      # score S1-S6 minimo per l'ensemble

# --------------------------------------------------------------
# CELLA 4 -- Lista strumenti
# --------------------------------------------------------------
# Formato: "Nome display": ("simbolo_TD", "exchange_o_None")
# ================================================================
#  INSTRUMENTS -- Piano free Twelvedata (verificato dal test run)
# ================================================================
#  INCLUSI (32):
#    Forex (13): tutti funzionanti
#    Commodity (1): XAU/USD -- WTI e Brent richiedono piano Grow+
#    Indici (1): DAX unico verificato -- SPX/NDX/DJI/FTSE/AEX/
#                EuroStoxx/CAC richiedono Grow+ o simbolo non valido
#    Azioni US (15): tutte funzionanti (exchange NASDAQ/NYSE)
#    Azioni EU (2): ASML/SAP funzionanti (quotate su NASDAQ/NYSE)
#    Azioni Asia (8): chiuse al test -- da verificare
#
#  RIMOSSI (19 strumenti -- piano a pagamento o simbolo non valido):
#    WTI, Brent, SPX, NDX, DJI, FTSE, EuroStoxx50, CAC40, AEX,
#    LVMH, Shell, Stellantis, BP, HSBC, TotalEnergies,
#    Santander, Nestle, Siemens, Schneider Electric
# ================================================================
INSTRUMENTS = {
    # ---- FOREX ----
    "EUR/USD":  ("EUR/USD",  None),  "USD/JPY":  ("USD/JPY",  None),
    "GBP/USD":  ("GBP/USD",  None),  "USD/CHF":  ("USD/CHF",  None),
    "AUD/USD":  ("AUD/USD",  None),  "EUR/GBP":  ("EUR/GBP",  None),
    "EUR/CHF":  ("EUR/CHF",  None),  "GBP/CHF":  ("GBP/CHF",  None),
    "EUR/CAD":  ("EUR/CAD",  None),  "EUR/AUD":  ("EUR/AUD",  None),
    "EUR/JPY":  ("EUR/JPY",  None),  "CAD/JPY":  ("CAD/JPY",  None),
    "CHF/JPY":  ("CHF/JPY",  None),
    # ---- COMMODITY ----
    "Oro (XAU/USD)": ("XAU/USD", None),
    # ---- INDICI ----
    "DAX 40": ("DAX", None),
    # ---- AZIONI US ----
    "Apple (AAPL)":          ("AAPL",  "NASDAQ"),
    "Microsoft (MSFT)":      ("MSFT",  "NASDAQ"),
    "NVIDIA (NVDA)":         ("NVDA",  "NASDAQ"),
    "Tesla (TSLA)":          ("TSLA",  "NASDAQ"),
    "Amazon (AMZN)":         ("AMZN",  "NASDAQ"),
    "Meta (META)":           ("META",  "NASDAQ"),
    "Alphabet (GOOGL)":      ("GOOGL", "NASDAQ"),
    "Bank of America (BAC)": ("BAC",   "NYSE"),
    "Intel (INTC)":          ("INTC",  "NASDAQ"),
    "JPMorgan (JPM)":        ("JPM",   "NYSE"),
    "Visa (V)":              ("V",     "NYSE"),
    "Mastercard (MA)":       ("MA",    "NYSE"),
    "P&G (PG)":              ("PG",    "NYSE"),
    "Coca-Cola (KO)":        ("KO",    "NYSE"),
    "Berkshire B (BRK.B)":   ("BRK.B", "NYSE"),
    # ---- AZIONI EU (quotate NASDAQ/NYSE -- piano free OK) ----
    "ASML": ("ASML", "NASDAQ"),
    "SAP":  ("SAP",  "NYSE"),
    # ---- AZIONI ASIATICHE (chiuse al test -- da verificare) ----
    "Mitsubishi UFJ (8306)": ("8306",    "TSE"),
    "SoftBank (9984)":       ("9984",    "TSE"),
    "Toyota (7203)":         ("7203",    "TSE"),
    "Bank of China (3988)":  ("3988",    "HKEX"),
    "Sony (6758)":           ("6758",    "TSE"),
    "Alibaba (9988)":        ("9988",    "HKEX"),
    "Tencent (0700)":        ("700",     "HKEX"),
    "BYD (002594)":          ("002594",  "SZSE"),
}

#==============================================================

# -- Strumenti + sessioni + market_status

SESSIONS = {"tokyo": (2, 9), "europe": (9, 17), "us": (15, 22)}
STOCK_MARKETS = {
    "tokyo":  ["8306","9984","7203","3988","6758","9988","700","002594"],
    "europe": ["ASML","SAP"],
    "us":     ["AAPL","MSFT","NVDA","TSLA","AMZN","META","GOOGL",
               "BAC","INTC","JPM","V","MA","PG","KO","BRK.B"],
}

# --------------------------------------------------------------
# CELLA 5 -- Funzioni di supporto (sessione, fetch API)
# --------------------------------------------------------------
def market_status(symbol, now_rome):
    hour = now_rome.hour + now_rome.minute / 60
    for mkt, syms in STOCK_MARKETS.items():
        if symbol in syms:
            s, e = SESSIONS[mkt]
            return "OPEN" if s <= hour < e else "CLOSED"
    return "OPEN"

def get_candles(symbol, exchange, interval, n):
    params = {"symbol": symbol, "interval": interval, "outputsize": n,
              "apikey": API_KEY, "format": "JSON", "order": "ASC"}
    if exchange:
        params["exchange"] = exchange
    for attempt in range(2):
        try:
            r    = requests.get(BASE_URL, params=params, timeout=20)
            data = r.json()
        except Exception as e:
            print(f"\n    [RETE] {symbol} {interval}: {e}", flush=True)
            return None, f"RETE: {e}"
        if data.get("status") == "error" or "values" not in data:
            code = data.get("code", "?")
            msg  = data.get("message", str(data)[:80])
            if code == 429:
                if attempt == 0:
                    print(f"\n    [RATE LIMIT] attendo 65s...", end="", flush=True)
                    time.sleep(65); continue
                return None, "RATE LIMIT 429"
            label = {400:"SIMBOLO NON TROVATO", 401:"API KEY NON VALIDA",
                     403:"PIANO NON SUPPORTATO", 404:"DATI NON DISPONIBILI"}.get(code, f"ERR {code}")
            print(f"\n    [{label}] {symbol}/{exchange} {interval}: {msg}", flush=True)
            return None, label
        try:
            df = pd.DataFrame(data["values"]).rename(columns={"datetime": "time"})
            for col in ["open","high","low","close"]:
                df[col] = pd.to_numeric(df[col], errors="coerce")
            df = df[["time","open","high","low","close"]].dropna()
            return (df.reset_index(drop=True) if len(df) >= 8 else None,
                    "OK" if len(df) >= 8 else f"CANDELE INSUFFICIENTI ({len(df)})")
        except Exception as e:
            return None, f"PARSE: {e}"
    return None, "RETRY ESAURITO"

#==============================================================

# -- Fetch API

def get_rsi_td(symbol, exchange, period=14):
    params = {"symbol": symbol, "interval": "1h", "time_period": period,
              "outputsize": 1, "apikey": API_KEY, "format": "JSON"}
    if exchange:
        params["exchange"] = exchange
    try:
        r    = requests.get("https://api.twelvedata.com/rsi", params=params, timeout=15)
        data = r.json()
        time.sleep(REQUEST_DELAY)
        if data.get("status") == "error" or "values" not in data:
            return None
        return round(float(data["values"][0]["rsi"]), 2)
    except Exception:
        return None

# --------------------------------------------------------------
# CELLA 6 -- Indicatori tecnici
# --------------------------------------------------------------
def calc_ema(closes, period):
    arr = np.asarray(closes, dtype=float)
    if len(arr) < period: return None
    k = 2.0 / (period + 1)
    ema = float(np.mean(arr[:period]))
    for p in arr[period:]: ema = p * k + ema * (1.0 - k)
    return round(ema, 6)

def calc_rsi(df, period=14):
    closes = df["close"].values
    if len(closes) < period + 1: return None
    deltas = np.diff(closes)
    gains = np.where(deltas > 0, deltas, 0.0)
    losses = np.where(deltas < 0, -deltas, 0.0)
    ag = np.mean(gains[:period]); al = np.mean(losses[:period])
    for i in range(period, len(deltas)):
        ag = (ag*(period-1)+gains[i])/period
        al = (al*(period-1)+losses[i])/period
    return round(100.0 if al == 0 else 100 - 100 / (1 + ag/al), 2)

#==============================================================

# -- Indicatori tecnici

def calc_rsi_series(closes, period=14):
    arr = np.asarray(closes, dtype=float)
    if len(arr) < period + 1: return np.array([])
    deltas = np.diff(arr)
    gains = np.where(deltas > 0, deltas, 0.0)
    losses = np.where(deltas < 0, -deltas, 0.0)
    ag = np.mean(gains[:period]); al = np.mean(losses[:period])
    out = []
    for i in range(period, len(deltas)):
        ag = (ag*(period-1)+gains[i])/period
        al = (al*(period-1)+losses[i])/period
        rs = ag/al if al != 0 else 100
        out.append(100 - 100 / (1 + rs))
    return np.array(out)

def classify_trend(df, n=8):
    lows = df["low"].values[-n:]; highs = df["high"].values[-n:]
    hl = sum(lows[i] > lows[i-1] for i in range(1, len(lows)))
    lh = sum(highs[i] < highs[i-1] for i in range(1, len(highs)))
    if hl >= 5: return "Rialzista"
    if lh >= 5: return "Ribassista"
    return "Laterale"

def count_bounces(df, n=8):
    lows = df["low"].values[-n:]; highs = df["high"].values[-n:]
    hl = sum(lows[i] > lows[i-1] for i in range(1, len(lows)))
    lh = sum(highs[i] < highs[i-1] for i in range(1, len(highs)))
    return max(hl, lh)

def check_ema_h4(df):
    if df is None or len(df) < 10: return None, None, "n/d"
    closes = df["close"].values
    e50 = calc_ema(closes, 50) if len(closes) >= 50 else None
    e200 = calc_ema(closes, 200) if len(closes) >= 200 else None
    if e50 is None: return None, None, "n/d"
    price = closes[-1]
    if e200 is not None:
        align = ("long" if price > e50 and e50 > e200 else
                 "short" if price < e50 and e50 < e200 else "mista")
    else:
        align = ("long (solo EMA50)" if price > e50 else
                 "short (solo EMA50)" if price < e50 else "mista")
    return e50, e200, align

def check_ema_h1(df):
    if df is None or len(df) < 50: return None, None, "n/d"
    closes = df["close"].values
    e20 = calc_ema(closes, 20); e50 = calc_ema(closes, 50)
    if e20 is None or e50 is None: return None, None, "n/d"
    price = closes[-1]
    align = ("long" if price > e20 and e20 > e50 else
             "short" if price < e20 and e20 < e50 else "mista")
    return e20, e50, align

def get_ema20_m15(df):
    if df is None or len(df) < 20: return None
    return calc_ema(df["close"].values, 20)

def find_fibonacci_levels(df_h4, trend):
    if df_h4 is None or len(df_h4) < 10: return None, "FIBONACCI H4: DATI INSUFFICIENTI"
    candles = df_h4.tail(50).reset_index(drop=True)
    highs = candles["high"].values; lows = candles["low"].values
    n = len(highs); swing_h, swing_l = [], []
    for i in range(2, n-3):
        if highs[i] >= max(highs[max(0,i-2):i]) and highs[i] >= max(highs[i+1:i+3]):
            swing_h.append(highs[i])
        if lows[i] <= min(lows[max(0,i-2):i]) and lows[i] <= min(lows[i+1:i+3]):
            swing_l.append(lows[i])
    if not swing_h or not swing_l: return None, "FIBONACCI H4: SWING NON IDENTIFICABILE"
    sh = swing_h[-1]; sl = swing_l[-1]; r = sh - sl
    if r <= 0: return None, "FIBONACCI H4: RANGE SWING ZERO"
    if trend == "Rialzista":
        lvl = {k: round(sh-v*r, 6) for k,v in [("38.2",.382),("50.0",.500),("61.8",.618),("78.6",.786)]}
    else:
        lvl = {k: round(sl+v*r, 6) for k,v in [("38.2",.382),("50.0",.500),("61.8",.618),("78.6",.786)]}
    return lvl, "OK"

def find_fib_zone(price, fib_levels, tol_pct=0.05):
    if fib_levels is None: return "N/D"
    for zone, lvl in fib_levels.items():
        if abs(price - lvl) / max(abs(lvl), 1e-9) <= tol_pct: return zone + "%"
    return "N/D"

def check_fib_confluence(sr_level, fib_levels, tol_pct=0.05):
    if sr_level is None or fib_levels is None: return False, "N/D"
    for zone, lvl in fib_levels.items():
        if abs(sr_level - lvl) / max(abs(lvl), 1e-9) <= tol_pct: return True, zone + "%"
    return False, "N/D"

def detect_rsi_divergence(df, period=14):
    if df is None or len(df) < period + 8: return "nessuna"
    rsi_vals = calc_rsi_series(df["close"].values, period)
    if len(rsi_vals) < 8: return "nessuna"
    window = min(20, len(rsi_vals))
    prices = df["close"].values[-(window+1):]; rsi_w = rsi_vals[-window:]
    md = 3; p_lows, p_highs = [], []
    for i in range(md, len(prices)-md):
        if all(prices[i] <= prices[j] for j in range(i-md,i+md+1) if j!=i): p_lows.append(i)
        if all(prices[i] >= prices[j] for j in range(i-md,i+md+1) if j!=i): p_highs.append(i)
    offset = len(prices) - len(rsi_w)
    if len(p_lows) >= 2:
        i1,i2 = p_lows[-2],p_lows[-1]; r1,r2 = i1-offset,i2-offset
        if 0<=r1<len(rsi_w) and 0<=r2<len(rsi_w) and prices[i2]<prices[i1] and rsi_w[r2]>rsi_w[r1]:
            return "rialzista"
    if len(p_highs) >= 2:
        i1,i2 = p_highs[-2],p_highs[-1]; r1,r2 = i1-offset,i2-offset
        if 0<=r1<len(rsi_w) and 0<=r2<len(rsi_w) and prices[i2]>prices[i1] and rsi_w[r2]<rsi_w[r1]:
            return "ribassista"
    return "nessuna"

def find_sr_level(df, trend, tol=0.002):
    if trend == "Laterale": return None, 0
    pivots = (df.tail(30)["low"].values if trend == "Rialzista" else df.tail(30)["high"].values)
    best, best_n = None, 0
    for p in pivots:
        cluster = [x for x in pivots if abs(x-p)/max(abs(p),1e-9) <= tol]
        if len(cluster) > best_n:
            best_n = len(cluster); best = round(float(np.mean(cluster)), 6)
    return best, best_n

def detect_pinbar(df_m15, trend, level, ema20_m15=None, prox=0.003):
    if level is None: return None
    for _, row in df_m15.tail(5).iloc[::-1].iterrows():
        o,h,l,c = row["open"],row["high"],row["low"],row["close"]
        body=abs(c-o); rng=h-l
        if rng == 0: continue
        uw=h-max(o,c); lw=min(o,c)-l
        near_lvl = abs((l if trend=="Rialzista" else h)-level)/max(abs(level),1e-9) <= prox
        if not near_lvl: continue
        if trend=="Rialzista" and lw>=rng*0.6 and body<=rng*0.30:
            near_ema = (ema20_m15 is not None and abs(c-ema20_m15)/max(abs(ema20_m15),1e-9)<=0.005)
            return {"type":"rialzista","entry":round(c,6),"pin_low":round(l,6),
                    "pin_high":round(h,6),"level_type":"supporto","near_ema20_m15":near_ema}
        if trend=="Ribassista" and uw>=rng*0.6 and body<=rng*0.30:
            near_ema = (ema20_m15 is not None and abs(c-ema20_m15)/max(abs(ema20_m15),1e-9)<=0.005)
            return {"type":"ribassista","entry":round(c,6),"pin_low":round(l,6),
                    "pin_high":round(h,6),"level_type":"resistenza","near_ema20_m15":near_ema}
    return None

def calc_sl_tp(pinbar, level, trend, rr=RR_MIN):
    buf = 0.001
    if trend == "Rialzista":
        sl = round(min(pinbar["pin_low"], level) * (1-buf), 6)
        tp = round(pinbar["entry"] + rr * (pinbar["entry"]-sl), 6)
    else:
        sl = round(max(pinbar["pin_high"], level) * (1+buf), 6)
        tp = round(pinbar["entry"] - rr * (sl-pinbar["entry"]), 6)
    return sl, tp

def validate_data(df_h4, df_h1, df_m15):
    MIN_C = {"H4": 30, "H1": 50, "M15": 20}
    for label, df in [("H4",df_h4),("H1",df_h1),("M15",df_m15)]:
        if df is None or len(df) < MIN_C[label]:
            return False, [f"SERIE {label} INSUFFICIENTE"], 0
        bad = df[(df["high"]<df["low"])|(df["open"]>df["high"])|(df["open"]<df["low"])|
                 (df["close"]>df["high"])|(df["close"]<df["low"])|(df["close"]==0)]
        if len(bad) > 0: return False, [f"OHLC CORROTTI ({label})"], 0
        if df[["open","high","low","close"]].isnull().any().any():
            return False, [f"NaN PRESENTI ({label})"], 0
    return True, ["OK"], 0

# --------------------------------------------------------------
# CELLA 7 -- Colonne output e funzione analyze
# --------------------------------------------------------------
COLUMNS = [
    "Nome prodotto","Data e ora analisi","Piattaforma","Flag qualit? dati",
    "Trend H4","EMA H4","Fibonacci H4 zona",
    "Trend H1","Allineamento H4-H1","EMA H1",
    "Divergenza RSI H1","Confluenza Fib+S/R H1",
    "Livello S/R H1","Livello H1 >= 2 tocchi?",
    "Pinbar M15 valida","RSI M15 ok?",
    "Gate superati",
    "Direzione","Entry","SL","TP","R:R effettivo",
    "TP >= 2.5x SL?","SL oltre il livello?",
    "Score qualit?","Raccomandazione size","Setup descrizione",
    "RSI H1","Rimbalzi ult. 8 H1","Timeframe",
]

def analyze(name, symbol, exchange, now_rome):
    res = {k: "--" for k in COLUMNS}
    res.update({"Nome prodotto": name,
                "Data e ora analisi": now_rome.strftime("%Y-%m-%d %H:%M"),
                "Piattaforma": PLATFORM, "Timeframe": "H4+H1+M15",
                "Direzione": "nessun setup", "Gate superati": "--",
                "Score qualit?": "--", "R:R effettivo": f"min 1:{RR_MIN}"})

    if market_status(symbol, now_rome) == "CLOSED":
        for k in COLUMNS:
            if k not in ("Nome prodotto","Data e ora analisi","Piattaforma","Timeframe"):
                res[k] = "mercato chiuso"
        res["Piattaforma"] = PLATFORM; res["Timeframe"] = "H4+H1+M15"
        return res

    df_h4,  err_h4  = get_candles(symbol, exchange, "4h",   250)
    time.sleep(REQUEST_DELAY)
    df_h1,  err_h1  = get_candles(symbol, exchange, "1h",   100)
    time.sleep(REQUEST_DELAY)
    df_m15, err_m15 = get_candles(symbol, exchange, "15min", 30)
    time.sleep(REQUEST_DELAY)

    ok, flags, penalty = validate_data(df_h4, df_h1, df_m15)
    if not ok:
        causes = " | ".join(filter(None, [
            f"H4:{err_h4}"   if df_h4  is None else None,
            f"H1:{err_h1}"   if df_h1  is None else None,
            f"M15:{err_m15}" if df_m15 is None else None,
        ]))
        flags = [f"{flags[0]}" + (f" [{causes}]" if causes else "")]
    res["Flag qualit? dati"] = "; ".join(flags)
    if not ok:
        res["Gate superati"] = f"G1x ({flags[0]})"; res["Direzione"] = "dati non disponibili"
        return res

    trend_h4 = classify_trend(df_h4, n=8); _, _, ema_h4_align = check_ema_h4(df_h4)
    res["Trend H4"] = trend_h4; res["EMA H4"] = ema_h4_align
    if trend_h4 == "Laterale" and ema_h4_align == "mista":
        res["Gate superati"] = "G1v G2x (BIAS H4 NON DEFINITO)"; return res
    trend_h4_bias = (trend_h4 if trend_h4 != "Laterale"
                     else ("Rialzista" if "long" in ema_h4_align else "Ribassista"))

    fib_levels, fib_msg = find_fibonacci_levels(df_h4, trend_h4_bias)

    trend_h1 = classify_trend(df_h1, n=8); res["Trend H1"] = trend_h1
    conflict = ((trend_h4_bias=="Rialzista" and trend_h1=="Ribassista") or
                (trend_h4_bias=="Ribassista" and trend_h1=="Rialzista"))
    if conflict:
        res["Allineamento H4-H1"] = "No -- CONFLITTO"
        res["Gate superati"] = "G1v G2v G3x (CONFLITTO H4/H1)"; return res
    if trend_h1 == "Laterale":
        res["Allineamento H4-H1"] = "No -- H1 LATERALE"
        res["Gate superati"] = "G1v G2v G3x (H1 LATERALE)"; return res
    res["Allineamento H4-H1"] = "S?"; trend = trend_h1

    _, _, ema_h1_align = check_ema_h1(df_h1); res["EMA H1"] = ema_h1_align
    rsi_h1 = get_rsi_td(symbol, exchange)
    if rsi_h1 is None: rsi_h1 = calc_rsi(df_h1)
    res["RSI H1"] = rsi_h1 if rsi_h1 is not None else "n/d"
    div_h1 = detect_rsi_divergence(df_h1); res["Divergenza RSI H1"] = div_h1
    res["Rimbalzi ult. 8 H1"] = count_bounces(df_h1)

    level, n_t = find_sr_level(df_h1, trend); level_ok = level is not None and n_t >= 2
    res["Livello S/R H1"] = level if level else "N/D"
    res["Livello H1 >= 2 tocchi?"] = "S?" if level_ok else "No"
    if not level_ok:
        res["Gate superati"] = "G1v G2v G3v G4x (NESSUN LIVELLO H1 VALIDO)"; return res

    price_now = float(df_h1["close"].values[-1])
    fib_zone  = find_fib_zone(price_now, fib_levels)
    res["Fibonacci H4 zona"] = fib_zone if fib_levels else fib_msg
    fib_confl, fib_confl_zone = check_fib_confluence(level, fib_levels)
    res["Confluenza Fib+S/R H1"] = ("S? ("+fib_confl_zone+")" if fib_confl else "No")

    ema20_m15 = get_ema20_m15(df_m15)
    pinbar    = detect_pinbar(df_m15, trend, level, ema20_m15)
    res["Pinbar M15 valida"] = "S?" if pinbar else "No"
    if not pinbar:
        res["Gate superati"] = "G1v G2v G3v G4v G5x (PINBAR M15 ASSENTE)"; return res

    rsi_m15 = calc_rsi(df_m15); rsi_m15_ok = False
    if rsi_m15 is not None and not (rsi_m15 > 75 or rsi_m15 < 25):
        if trend == "Rialzista" and rsi_m15 <= 70: rsi_m15_ok = True
        if trend == "Ribassista" and rsi_m15 >= 30: rsi_m15_ok = True
    res["RSI M15 ok?"] = "S?" if rsi_m15_ok else "No"

    sl, tp   = calc_sl_tp(pinbar, level, trend, RR_MIN)
    entry    = pinbar["entry"]; dist_sl = abs(entry-sl); dist_tp = abs(tp-entry)
    rr_actual = round(dist_tp/dist_sl, 2) if dist_sl > 0 else 0.0
    rr_ok    = rr_actual >= (RR_MIN - 0.05)
    sl_ok    = ((trend=="Rialzista" and sl<level) or (trend=="Ribassista" and sl>level))
    res["Entry"] = entry; res["SL"] = sl; res["TP"] = tp
    res["R:R effettivo"] = f"1:{rr_actual}"
    res["TP >= 2.5x SL?"] = "S?" if rr_ok else "No"
    res["SL oltre il livello?"] = "S?" if sl_ok else "No"
    if not rr_ok:
        res["Gate superati"] = "G1v G2v G3v G4v G5v G6x (R:R INSUFFICIENTE)"; return res

    res["Gate superati"] = "G1v G2v G3v G4v G5v G6v"

    def dir_ok(align):
        return (("long" in align and trend=="Rialzista") or ("short" in align and trend=="Ribassista"))
    score, det = 0, []
    if ema_h4_align != "n/d" and dir_ok(ema_h4_align): score+=1; det.append("S1v")
    else: det.append("S1x")
    if ema_h1_align != "n/d" and dir_ok(ema_h1_align): score+=1; det.append("S2v")
    else: det.append("S2x")
    if fib_confl and fib_confl_zone in ("50.0%","61.8%"): score+=1; det.append("S3v")
    else: det.append("S3x")
    if ((trend=="Rialzista" and div_h1=="rialzista") or
            (trend=="Ribassista" and div_h1=="ribassista")): score+=1; det.append("S4v")
    else: det.append("S4x")
    if rsi_m15_ok: score+=1; det.append("S5v")
    else: det.append("S5x")
    if pinbar.get("near_ema20_m15"): score+=1; det.append("S6v")
    else: det.append("S6x")

    score_final = max(0, score - penalty)
    score_str   = f"{score_final}/6 [{' '.join(det)}]"
    size_rec = ("ALTA CONFLUENZA -- size piena" if score_final >= 5 else
                "CONFLUENZA MEDIA -- size 50%"  if score_final >= 3 else
                "BASSA CONFLUENZA -- non operare")
    setup_desc = (f"{'Supporto' if trend=='Rialzista' else 'Resistenza'} H1 ({level})"
                  +(f" + Fib {fib_confl_zone}" if fib_confl else "")
                  +f" + pinbar M15 {pinbar['type']}"
                  +(f" + divRSI H1 {div_h1}" if div_h1 != "nessuna" else "")
                  +(f" + EMA20 M15" if pinbar.get("near_ema20_m15") else ""))
    res.update({"Direzione": "buy" if trend=="Rialzista" else "sell",
                "Score qualit?": score_str, "Raccomandazione size": size_rec,
                "Setup descrizione": setup_desc})
    return res

#==============================================================


# --------------------------------------------------------------
# CELLA 8 -- Export Excel analisi TD
# --------------------------------------------------------------
HDR_FILL=PatternFill("solid",fgColor="1F4E79"); BULL_FILL=PatternFill("solid",fgColor="C6EFCE")
BEAR_FILL=PatternFill("solid",fgColor="FFC7CE"); LAT_FILL=PatternFill("solid",fgColor="FFEB9C")
CLOSED_FILL=PatternFill("solid",fgColor="D9D9D9"); YES_FILL=PatternFill("solid",fgColor="E2EFDA")
NO_FILL=PatternFill("solid",fgColor="FCE4D6"); SCORE_HI=PatternFill("solid",fgColor="70AD47")
SCORE_MD=PatternFill("solid",fgColor="FFD966"); SCORE_LO=PatternFill("solid",fgColor="FF7B7B")
BORDER=Border(left=Side(style="thin"),right=Side(style="thin"),
              top=Side(style="thin"),bottom=Side(style="thin"))
SI_NO_COLS={"Livello H1 >= 2 tocchi?","Pinbar M15 valida","RSI M15 ok?",
            "TP >= 2.5x SL?","SL oltre il livello?","Allineamento H4-H1"}
COL_WIDTHS_TD=[26,18,14,30,12,22,14,12,18,22,18,20,14,16,14,10,44,
               10,14,14,14,12,14,18,40,28,58,8,10,12]

def export_excel_td(rows, path):
    wb = openpyxl.Workbook(); ws = wb.active; ws.title = "Analisi H4+H1+M15"
    for ci,col in enumerate(COLUMNS,1):
        c=ws.cell(row=1,column=ci,value=col)
        c.font=Font(bold=True,color="FFFFFF",name="Arial",size=10)
        c.fill=HDR_FILL; c.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
        c.border=BORDER
    ws.row_dimensions[1].height=52
    for ri,row in enumerate(rows,2):
        trend=str(row.get("Trend H1",""));closed=str(row.get("Direzione",""))=="mercato chiuso"
        for ci,col in enumerate(COLUMNS,1):
            val=row.get(col,""); c=ws.cell(row=ri,column=ci,value=val)
            c.font=Font(name="Arial",size=10)
            c.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
            c.border=BORDER
            if   closed:               c.fill=CLOSED_FILL
            elif "Rialzista" in trend: c.fill=BULL_FILL
            elif "Ribassista" in trend:c.fill=BEAR_FILL
            elif "Laterale" in trend:  c.fill=LAT_FILL
            if col in SI_NO_COLS:
                v=str(val)
                if v=="S?" or v.startswith("S?"):
                    c.fill=YES_FILL; c.font=Font(name="Arial",size=10,color="375623",bold=True)
                elif v.startswith("No"):
                    c.fill=NO_FILL;  c.font=Font(name="Arial",size=10,color="9C0006",bold=True)
            if col=="Score qualit?" and "/" in str(val):
                try:
                    s=int(str(val).split("/")[0])
                    c.fill=(SCORE_HI if s>=5 else SCORE_MD if s>=3 else SCORE_LO)
                    if s>=5: c.font=Font(name="Arial",size=10,bold=True)
                except ValueError: pass
    for i,w in enumerate(COL_WIDTHS_TD,1):
        ws.column_dimensions[get_column_letter(i)].width=w
    ws.freeze_panes="A2"; ws.auto_filter.ref=ws.dimensions
    wb.save(path); print(f"OK Analisi TD salvata: {os.path.basename(path)}")

# --------------------------------------------------------------
# CELLA 9 -- Funzioni Ensemble
# --------------------------------------------------------------
def _parse_score_ens(val):
    matches = re.findall(r'(\d+)/6', str(val))
    return int(matches[-1]) if matches else None

def _parse_rr_ens(val):
    m = re.search(r'1:(\d+\.?\d*)', str(val))
    return float(m.group(1)) if m else None

def calc_composite(row):
    total = 0.0; parts = {}
    s = _parse_score_ens(row.get('Score qualit?', ''))
    c1 = round((s/6)*60, 1) if s is not None else 0.0
    parts['S1-S6'] = c1; total += c1
    rr = _parse_rr_ens(row.get('R:R effettivo', ''))
    c2 = round(min((rr-2.5)/2.5*15, 15), 1) if rr and rr >= 2.5 else 0.0
    parts['R:R'] = c2; total += c2
    try: b = int(float(row.get('Rimbalzi ult. 8 H1', 0) or 0))
    except: b = 0
    c3 = round(min((b-2)/3*10, 10), 1) if b >= 2 else 0.0
    parts['Rimb'] = c3; total += c3
    fib = str(row.get('Fibonacci H4 zona', ''))
    c4 = 5.0 if any(z in fib for z in ['50.0%','61.8%']) else 2.0 if any(z in fib for z in ['38.2%','78.6%']) else 0.0
    parts['Fib'] = c4; total += c4
    detail = f"S1-S6:{parts['S1-S6']} | R:R:{parts['R:R']} | Rimb:{parts['Rimb']} | Fib:{parts['Fib']} | Cross:0"
    return round(min(total, 90.0), 1), detail

#==============================================================

# -- Export Excel TD + costanti stile

def apply_cross_platform(df):
    df = df.copy()
    df['_k'] = df['Nome prodotto'].str.lower().str.strip() + '|' + df['Direzione'].str.lower().str.strip()
    src_count = df.groupby('_k')['Sorgente'].nunique()
    def _upd(row):
        n = src_count.get(row['_k'], 1)
        if n >= 2:
            row['Score confluenza (%)'] = min(round(row['Score confluenza (%)']+10, 1), 100)
            row['Cross-platform'] = f'v ({n} sorgenti)'
            row['Dettaglio score'] = row['Dettaglio score'].replace('Cross:0','Cross:10')
        else:
            row['Cross-platform'] = '--'
        return row
    return df.apply(_upd, axis=1).drop(columns=['_k'])

ENS_COLS = ['Score confluenza (%)','Cross-platform','Sorgente','Nome prodotto','Direzione',
            'Entry','SL','TP','R:R effettivo','Gate superati','Score qualit?',
            'EMA H4','EMA H1','Fibonacci H4 zona','Confluenza Fib+S/R H1',
            'Divergenza RSI H1','Pinbar M15 valida','RSI M15 ok?','RSI H1',
            'Rimbalzi ult. 8 H1','SL oltre il livello?','Raccomandazione size',
            'Dettaglio score','Setup descrizione','Piattaforma','Data e ora analisi']
ENS_WIDTHS = {'Score confluenza (%)':14,'Cross-platform':18,'Sorgente':12,'Nome prodotto':28,
              'Direzione':10,'Entry':14,'SL':14,'TP':14,'R:R effettivo':12,
              'Gate superati':42,'Score qualit?':40,'EMA H4':22,'EMA H1':22,
              'Fibonacci H4 zona':14,'Confluenza Fib+S/R H1':20,'Divergenza RSI H1':16,
              'Pinbar M15 valida':14,'RSI M15 ok?':10,'RSI H1':8,'Rimbalzi ult. 8 H1':12,
              'SL oltre il livello?':16,'Raccomandazione size':28,'Dettaglio score':44,
              'Setup descrizione':55,'Piattaforma':26,'Data e ora analisi':18}

HI_FILL=PatternFill("solid",fgColor="70AD47"); MD_FILL=PatternFill("solid",fgColor="FFD966")
LO_FILL_ENS=PatternFill("solid",fgColor="FFC7CE"); XPLT_FILL=PatternFill("solid",fgColor="BDD7EE")

def load_source_file(path, label):
    """Carica un file di analisi, restituisce solo i setup con tutti i gate superati."""
    import re as _re2
    for sheet in ('Analisi H4+H1+M15','Analisi H1+M15'):
        try: df = pd.read_excel(path, sheet_name=sheet); break
        except: continue
    else: return None
    df = df[df['Direzione'].astype(str).str.strip().str.lower().isin(['buy','sell'])].copy()
    if df.empty: return None
    if 'Gate superati' in df.columns:
        df = df[df['Gate superati'].astype(str).str.contains(r'G6[vv]', na=False, regex=True)]
    if df.empty: return None
    if 'Score qualita?' in df.columns or 'Score qualit?' in df.columns:
        col = 'Score qualita?' if 'Score qualita?' in df.columns else 'Score qualit?'
    else:
        col = 'Score qualita?' if 'Score qualita?' in df.columns else 'Score qualit?'
    score_col = next((c for c in df.columns if 'Score qualit' in c), None)
    if score_col:
        def _s(v):
            m = _re2.findall(r'(\d+)/6', str(v))
            return int(m[-1]) if m else 0
        df = df[df[score_col].apply(_s) >= SCORE_MIN_ENS]
    if df.empty: return None
    df['Sorgente'] = label
    return df


def build_ensemble(all_dfs):
    """
    Accetta: un singolo DataFrame O una lista di DataFrame O un dict {label: DataFrame}.
    Calcola score composito, applica bonus cross-platform.
    """
    import re as _re3
    # Normalizza input
    if isinstance(all_dfs, pd.DataFrame):
        dfs = [all_dfs]
    elif isinstance(all_dfs, dict):
        dfs = [df for df in all_dfs.values() if df is not None]
    else:
        dfs = [df for df in all_dfs if df is not None]
    if not dfs: return pd.DataFrame()

    combined = pd.concat(dfs, ignore_index=True)
    # Trova la colonna Score qualita (gestisce varianti encoding)
    score_col = next((c for c in combined.columns if 'Score qualit' in c), 'Score qualita?')

    rows = []
    for _, r in combined.iterrows():
        if str(r.get('Direzione','')).strip().lower() not in ('buy','sell'): continue
        gate = str(r.get('Gate superati',''))
        if not _re3.search(r'G6[vv]', gate): continue
        s_raw = str(r.get(score_col,''))
        m = _re3.findall(r'(\d+)/6', s_raw)
        s = int(m[-1]) if m else None
        if s is None or s < SCORE_MIN_ENS: continue
        sc, detail = calc_composite(r)
        row = {'Score confluenza (%)': sc, 'Cross-platform': '--', 'Dettaglio score': detail,
               'Sorgente': str(r.get('Sorgente','Twelvedata'))}
        for col in ENS_COLS:
            if col not in row: row[col] = r.get(col, '--')
        rows.append(row)

    if not rows: return pd.DataFrame()
    out = pd.DataFrame(rows)
    out = apply_cross_platform(out)
    out = out.sort_values('Score confluenza (%)', ascending=False).reset_index(drop=True)
    return out[[c for c in ENS_COLS if c in out.columns]]

#==============================================================

# -- Funzioni Ensemble (build, export, load)


def _write_ens_sheet(ws, df):
    """Scrive DataFrame su foglio Excel con stile ensemble (riusato per tutti i fogli)."""
    cols = list(df.columns)
    for ci,col in enumerate(cols,1):
        c=ws.cell(row=1,column=ci,value=col)
        c.font=Font(bold=True,color="FFFFFF",name="Arial",size=10)
        c.fill=HDR_FILL; c.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
        c.border=BORDER
    ws.row_dimensions[1].height=50
    sc_ci   = cols.index('Score confluenza (%)')+1 if 'Score confluenza (%)' in cols else None
    xplt_ci = cols.index('Cross-platform')+1      if 'Cross-platform' in cols else None
    for ri,(_,row) in enumerate(df.iterrows(),2):
        score=float(row.get('Score confluenza (%)',0) or 0)
        dir_=str(row.get('Direzione','')).upper()
        cross=str(row.get('Cross-platform',''))
        base=BULL_FILL if dir_=='BUY' else BEAR_FILL if dir_=='SELL' else None
        for ci,col in enumerate(cols,1):
            val=row.get(col,''); c=ws.cell(row=ri,column=ci,value=val)
            c.font=Font(name="Arial",size=10)
            c.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
            c.border=BORDER
            if base: c.fill=base
        if sc_ci:
            sc=ws.cell(row=ri,column=sc_ci)
            sc.fill=HI_FILL if score>=75 else MD_FILL if score>=55 else LO_FILL_ENS
            if score>=75: sc.font=Font(name="Arial",size=10,bold=True)
        if xplt_ci and 'v' in cross:
            xc=ws.cell(row=ri,column=xplt_ci)
            xc.fill=XPLT_FILL; xc.font=Font(name="Arial",size=10,bold=True,color="1F4E79")
    for ci,col in enumerate(cols,1):
        ws.column_dimensions[get_column_letter(ci)].width=ENS_WIDTHS.get(col,14)
    ws.freeze_panes="A2"; ws.auto_filter.ref=ws.dimensions


def export_excel_ensemble(df, path):
    """
    Produce Excel con 5 fogli (stesso formato file ensemble di riferimento):
      Ensemble   - tutti i setup ordinati per score
      MT5        - solo setup MetaTrader5
      Twelvedata - solo setup Twelvedata
      yfinance   - solo setup yfinance
      Legenda    - spiegazione formula score
    """
    wb = openpyxl.Workbook()
    # Foglio principale
    ws_ens = wb.active; ws_ens.title = "Ensemble"
    _write_ens_sheet(ws_ens, df)
    # Fogli per sorgente (solo se presenti)
    for src_name in ('MT5','Twelvedata','yfinance'):
        sub = df[df['Sorgente'].astype(str)==src_name].reset_index(drop=True) \
              if 'Sorgente' in df.columns else pd.DataFrame()
        if not sub.empty:
            ws_src = wb.create_sheet(title=src_name)
            _write_ens_sheet(ws_src, sub)
    # Legenda
    ws_leg = wb.create_sheet('Legenda')
    ws_leg.column_dimensions['A'].width=32; ws_leg.column_dimensions['B'].width=55
    legenda=[
        ("LEGENDA SCORE COMPOSITO",""),("",""),
        ("Soglie colore Excel",""),
        ("Verde (>= 75)","Alta confluenza -- condizioni ottimali"),
        ("Giallo (55-74)","Confluenza media -- valutare"),
        ("Rosso (< 55)","Bassa confluenza -- sotto soglia operativa"),
        ("",""),("FORMULA (max 100 pt)",""),
        ("S1-S6 normalizzato (60 pt)","Score da script analisi / 6 x 60"),
        ("R:R effettivo (15 pt)","Scala lineare: 1:2.5=0pt, 1:5.0=15pt"),
        ("Rimbalzi S/R (10 pt)","Scala: 2=3pt, 3=6pt, 5+=10pt"),
        ("Zona Fibonacci (5 pt)","50%/61.8%=5pt, 38.2%/78.6%=2pt"),
        ("Cross-platform (10 pt)","+10 se stesso setup su 2+ sorgenti"),
        ("",""),("(!) DISCLAIMER",""),
        ("","Lo score NON e' una probabilita' statistica validata."),
        ("","Non esiste evidenza che uno score piu' alto"),
        ("","corrisponda a un tasso di successo piu' elevato."),
    ]
    for ri,(k,v) in enumerate(legenda,1):
        ca=ws_leg.cell(row=ri,column=1,value=k)
        cb=ws_leg.cell(row=ri,column=2,value=v)
        ca.font=Font(name="Arial",size=10,bold=(ri==1 or k in
            ("FORMULA (max 100 pt)","(!) DISCLAIMER","Soglie colore Excel")))
        cb.font=Font(name="Arial",size=10)
    sheets=[s.title for s in wb.worksheets]
    wb.save(path)
    print(f"[OK] Ensemble salvato: {os.path.basename(path)}  Fogli: {sheets}")


# --------------------------------------------------------------
# CELLA 10A -- ANALISI TD (chiama API Twelvedata)
# --------------------------------------------------------------
# Esegui questa cella per lanciare l'analisi su tutti gli strumenti.
# Al termine salva analisi_TD_H4H1M15_TIMESTAMP.xlsx in /content/
# --------------------------------------------------------------
print("="*58)
print("  TRADING ANALYSIS TD v3 -- Twelvedata")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("="*58)

# Test API key
print("\nVerifica API Key...", end=" ", flush=True)
try:
    _r = requests.get(BASE_URL,
        params={"symbol":"EUR/USD","interval":"1h","outputsize":1,"apikey":API_KEY},
        timeout=10)
    _d = _r.json()
    if _d.get("code") in (401,403) or (_d.get("status")=="error" and _d.get("code") != 404):
        raise ValueError(f"API Key non valida: {_d.get('message','')}")
    print("OK")
    time.sleep(REQUEST_DELAY)
except Exception as e:
    print(f"ERRORE: {e}")
    raise

# Analisi
now_utc  = datetime.now(timezone.utc)
now_rome = now_utc.astimezone(ROME_TZ)
n_open   = sum(1 for _,(sym,_) in INSTRUMENTS.items()
               if market_status(sym, now_rome) == "OPEN")
eta      = round(n_open * 4 * REQUEST_DELAY / 60, 1)
print(f"\nR:R minimo: 1:{RR_MIN} | Strumenti aperti: {n_open} | ETA: ~{eta} min\n")

results = []; total = len(INSTRUMENTS)
for i,(name,(symbol,exchange)) in enumerate(INSTRUMENTS.items(),1):
    print(f"  [{i:02d}/{total}] {name:<35} ({symbol})", end=" ... ", flush=True)
    row = analyze(name, symbol, exchange, now_rome)
    row['Sorgente'] = 'Twelvedata'
    results.append(row)
    print(row.get("Direzione","--"))

# Salva Excel
ts      = datetime.now().strftime('%Y%m%d_%H%M')
td_path = os.path.join(WORK_DIR, f"analisi_TD_H4H1M15_{ts}.xlsx")
export_excel_td(results, td_path)

# -- File intermedio (non richiede download: usato solo dalla Cella 10B) --------
print(f"\nFile intermedio salvato in sessione: {td_path}")
print("(File temporaneo -- solo l'ensemble finale viene salvato su Google Drive.)")

print("\nProseguo automaticamente con la Cella 10B (ensemble)...")

#==============================================================

# -- CELLA 10B -- Ensemble multi-sorgente (raccolta file)
# Esecuzione automatica: prosegue subito dopo la Cella 10A, nessun intervento manuale.
# Legge TUTTI i file di analisi disponibili in /content/:
#   analisi_MT5_H4H1M15_*.xlsx
#   analisi_TD_H4H1M15_*.xlsx
#   analisi_YF_H4H1M15_*.xlsx
# Puoi caricare manualmente file MT5/yfinance nel pannello Files (cartella sinistra)
# e rilanciare questa cella senza ripetere le chiamate API.
# --------------------------------------------------------------
import glob as _glob

_FILE_PATTERNS = {
    'MT5':        'analisi_MT5_H4H1M15_*.xlsx',
    'Twelvedata': 'analisi_TD_H4H1M15_*.xlsx',
    'yfinance':   'analisi_YF_H4H1M15_*.xlsx',
}

print("Ricerca file di analisi in /content/...")
_source_dfs = {}
for _label, _pat in _FILE_PATTERNS.items():
    _files = sorted(_glob.glob(os.path.join(WORK_DIR, _pat)))
    if not _files:
        print(f"  {_label:<12}: nessun file trovato")
        continue
    _latest = _files[-1]
    _df = load_source_file(_latest, _label)
    if _df is not None and not _df.empty:
        _source_dfs[_label] = _df
        print(f"  {_label:<12}: {len(_df)} setup da {os.path.basename(_latest)}")
    else:
        print(f"  {_label:<12}: 0 setup validi in {os.path.basename(_latest)}")

#==============================================================

# -- CELLA 10B -- Ensemble multi-sorgente

if not _source_dfs:
    print("\nNessun setup valido trovato in nessun file.")
    print("Possibili cause:")
    print("  - Nessun file di analisi in /content/ (esegui prima Cella 10A)")
    print("  - Tutti i gate non superati (nessuna pinbar M15, trend laterale, ecc.)")
    print("  - Errori di rete durante l'analisi (timeout API)")
else:
    print("\nCalcolo ensemble...")
    _ensemble = build_ensemble(list(_source_dfs.values()))

    if _ensemble.empty:
        print("Ensemble vuoto: nessun setup passa i filtri (gate + score >= 3/6).")
    else:
        _ts_ens  = datetime.now().strftime('%Y%m%d_%H%M')
        ens_path = os.path.join(DRIVE_ENSEMBLE_DIR, f"Ensemble_H4H1M15_{_ts_ens}.xlsx")
        export_excel_ensemble(_ensemble, ens_path)
        print(f"\nOK File ensemble salvato automaticamente su Google Drive: {ens_path}")

        hi = len(_ensemble[_ensemble['Score confluenza (%)'] >= 75])
        md = len(_ensemble[(_ensemble['Score confluenza (%)'] >= 55) & (_ensemble['Score confluenza (%)'] < 75)])
        print(f"\n{'='*58}")
        print(f"  Setup totali: {len(_ensemble)}  |  >=75: {hi}  |  55-74: {md}")
        print(f"{'='*58}")
        _pcols = ['Score confluenza (%)','Cross-platform','Sorgente','Nome prodotto',
                  'Direzione','R:R effettivo','Raccomandazione size']
        _pcols = [c for c in _pcols if c in _ensemble.columns]
        print(_ensemble[_pcols].to_string(index=False))

        print(f"\nUnico file di output: {os.path.basename(ens_path)}")
        print(f"Percorso Drive: {ens_path}")
        print("(Il file rimane anche disponibile in Google Drive dopo la chiusura del notebook.)")

print("\nNOTA: Score confluenza NON e' probabilita' statistica validata.")
